# 5. Resource Portfolio and Production Maps

**Pipeline position:** runs after `1_data_prep.ipynb`. Uses `Master.csv`
directly (the world map needs all countries, not just the 38-country sample).

## What this notebook does

Pre-computes four small tables for the descriptive and robustness charts:

| Chart | Table |
|---|---|
| 17 — forest vs extractive rent decomposition | rent_decomp_top20.csv |
| 20 — forest-adjusted vs original scatter | rent_adj_vs_orig.csv |
| 25 — resource portfolio composition (top-20 most diversified) | resource_portfolio.csv |
| 1 — production-per-capita world map | production_per_capita_world.csv |

Diversification is measured by Shannon entropy across the four rent
components (oil, gas, mineral, forestry). A country with a single dominant
rent type has low entropy; a country splitting rents evenly across all four
has the maximum. Top-20 keeps the most balanced.

## Outputs

- `rent_decomp_top20.csv` — Country, four rent components, Total_Rents
- `rent_adj_vs_orig.csv` — per-country Total_NR vs Adjusted_NR (2019)
- `resource_portfolio.csv` — top-20 most diversified countries with each
  rent component as a share of total
- `production_per_capita_world.csv` — Country Code, Country Name,
  production-value per capita (averaged 2015–2019), log10 colour value


In [1]:
import os, sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd

from _config import (INTERMEDIARY, ARTIFACTS, artifact_path)

master_full = pd.read_csv(os.path.join(INTERMEDIARY, 'Master.csv'))
print(f'Master: {master_full["Country Code"].nunique()} countries, {len(master_full):,} rows')


Master: 75 countries, 1,875 rows


## Chart 17 — top-20 producers, rent decomposition (2019)

Take the 2019 cross-section, sum the four rent components per country,
keep the top-20 by total. The viz notebook stacks the bars in fixed order.


In [2]:
rent_cols = [
    'Oil rents (% of GDP)',
    'Natural gas rents (% of GDP)',
    'Mineral rents (% of GDP)',
    'Forestry rents (% of GDP)',
]
short = {
    'Oil rents (% of GDP)':         'Oil',
    'Natural gas rents (% of GDP)': 'Gas',
    'Mineral rents (% of GDP)':     'Mineral',
    'Forestry rents (% of GDP)':    'Forestry',
}

latest = master_full[master_full['Year'] == 2019].copy()
for c in rent_cols:
    latest[c] = latest[c].fillna(0)
latest['Total_Rents'] = latest[rent_cols].sum(axis=1)

top20 = latest.nlargest(20, 'Total_Rents').sort_values('Total_Rents', ascending=True)
out = top20[['Country Name', 'Country Code'] + rent_cols + ['Total_Rents']].copy()
out.columns = ['Country Name', 'Country Code',
               'Oil', 'Gas', 'Mineral', 'Forestry', 'Total_Rents']
out.to_csv(artifact_path('rent_decomp_top20.csv'), index=False)
print(f'  Saved rent_decomp_top20.csv ({len(out)} rows)')


  Saved rent_decomp_top20.csv (20 rows)


## Chart 20 — total vs forest-adjusted rents (2019)

The diagonal in the chart is `Adjusted = Total`. Countries above the
diagonal are the forestry-driven outliers the sample-selection correction
filters out.


In [3]:
rc = master_full[master_full['Year'] == 2019][
    ['Country Code', 'Country Name',
     'Total natural resources rents (% of GDP)',
     'Forestry rents (% of GDP)']
].copy().dropna()
rc['Total_NR']    = rc['Total natural resources rents (% of GDP)']
rc['Adjusted_NR'] = (rc['Total_NR'] - rc['Forestry rents (% of GDP)'].fillna(0)).clip(lower=0)
rc[['Country Code', 'Country Name', 'Total_NR', 'Adjusted_NR']] \
    .to_csv(artifact_path('rent_adj_vs_orig.csv'), index=False)
print(f'  Saved rent_adj_vs_orig.csv ({len(rc)} rows)')


  Saved rent_adj_vs_orig.csv (75 rows)


## Chart 25 — resource portfolio composition

Use the average 2015–2019 rents per country to smooth out year-on-year
volatility. Compute Shannon entropy across the four rent shares; keep the
20 countries with the highest entropy. Output as shares (each row sums to
roughly 1.0, with tiny rounding).


In [4]:
recent = master_full[master_full['Year'].between(2015, 2019)].copy()
for c in rent_cols:
    recent[c] = recent[c].fillna(0)
agg = recent.groupby(['Country Code', 'Country Name'])[rent_cols].mean().reset_index()
agg['Total'] = agg[rent_cols].sum(axis=1)
agg = agg[agg['Total'] > 0.5]  # drop countries with negligible rents

for c in rent_cols:
    agg[short[c] + '_share'] = agg[c] / agg['Total']

shares = agg[[short[c] + '_share' for c in rent_cols]].values
# Shannon entropy (natural log), with safe handling of zeros.
ent = -np.sum(np.where(shares > 0, shares * np.log(shares + 1e-12), 0), axis=1)
agg['entropy'] = ent

top_div = agg.nlargest(20, 'entropy').sort_values('entropy', ascending=False).copy()
out = top_div[['Country Code', 'Country Name', 'Total', 'entropy'] +
              [short[c] + '_share' for c in rent_cols]].copy()
out.columns = ['Country Code', 'Country Name', 'Total_Rents', 'Entropy',
               'Oil', 'Gas', 'Mineral', 'Forestry']
out.to_csv(artifact_path('resource_portfolio.csv'), index=False)
print(f'  Saved resource_portfolio.csv ({len(out)} rows)')


  Saved resource_portfolio.csv (20 rows)


## Chart 1 — production-per-capita world map

Use the 2015–2019 average of `Total_Production_Value / Population` for
every country (no sample restriction — the map shows the whole world). Add
a log10 column for the choropleth colour scale.


In [5]:
prod = master_full[master_full['Year'].between(2015, 2019)][
    ['Country Code', 'Country Name', 'Total_Production_Value', 'Population']
].copy().dropna(subset=['Country Code'])

prod['prod_pc'] = prod['Total_Production_Value'] / prod['Population'].replace(0, np.nan)
agg = (prod.groupby(['Country Code', 'Country Name'], as_index=False)['prod_pc']
            .mean()
            .dropna())
agg = agg[agg['prod_pc'] > 0]
agg['log10_prod_pc'] = np.log10(agg['prod_pc'])
agg.to_csv(artifact_path('production_per_capita_world.csv'), index=False)
print(f'  Saved production_per_capita_world.csv ({len(agg)} countries)')


  Saved production_per_capita_world.csv (74 countries)


## Summary

| Artifact | Rows | Used by |
|---|---|---|
| rent_decomp_top20.csv | 20 | chart 17 |
| rent_adj_vs_orig.csv | one per country with rent data | chart 20 |
| resource_portfolio.csv | 20 most diversified countries | chart 25 |
| production_per_capita_world.csv | one per country | chart 1 |

This is the last prep notebook. Run any of the four `viz_*.ipynb` notebooks
to render the corresponding chart group.
